# TF-Mamba goc (dual-stream, paper) — Kaggle runner

Chay **TF-Mamba GOC** (chi co TF-Mamba, khong ablation) tren HUST-HAR / UT-HAR / NTU-Fi.

- **Model:** dual-stream Mamba + AdaptiveFusion + proj_s3 + GAP (paper Liu 2025).
- **Protocol:** AdamW lr=1e-4 (paper), wd=0.01, 40ep, bs=32, grad_clip=1.0, early-stop loss<0.01, no scheduler.
- **Split (giong git ho):** HUST random 80/20 (seed 42); UT-HAR official train/test (val bo, `MERGE_VAL` de gop); NTU-Fi train_amp/test_amp.
- **Norm:** HUST = data_norm; UT-HAR/NTU-Fi theo `NORM_MODE`:
  - `author` = SenseFi norm (min-max / hang-so) tren raw -> DWT (dung tac gia, KHONG z lai)
  - `double` = SenseFi norm -> DWT -> z-norm (2 lan)
- **Seed co dinh = 42** (1 lan chay).

**Truoc khi chay:** Add Input 3 dataset RAW (HUST / UT-HAR / NTU-Fi) + bat **GPU**.

In [ ]:
# Cell 1 — Install mamba-ssm + PyWavelets
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
!pip install -q PyWavelets
print('Install done')

In [ ]:
# Cell 2 — Clone code + import mrngoc (standalone, khong import xrf55_bench)
import sys, subprocess
from pathlib import Path

CODE = Path('/kaggle/working/xrf55_benchmark')
if not CODE.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/imhoangt/xrf55_benchmark.git', str(CODE)], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE), check=True)

sys.path.insert(0, str(CODE / 'mrngoc'))
import data, model, train
print('Import OK : mrngoc (model / data / train)')

In [ ]:
# Cell 3 — Config + tim mount cua tung dataset
from pathlib import Path

DATASETS  = ['hust', 'uthar', 'ntufi']   # bo bot neu chi muon chay 1 vai cai
SEED      = 42
NORM_MODE = 'author'   # 'author' (SenseFi norm, dung tac gia) | 'double' (+ z-norm)
MERGE_VAL = False      # CHI UT-HAR: False = git SenseFi (test=X_test) | True = gop val vao test
_MARKER   = {'hust': 'HUST_HAR_labels.pt', 'uthar': 'X_train.csv', 'ntufi': 'train_amp'}

def resolve_mount(ds):
    base = Path('/kaggle/input')
    for c in (sorted(base.iterdir()) if base.is_dir() else []):
        if next(c.rglob(_MARKER[ds]), None) is not None:
            return str(c)
    raise FileNotFoundError(f'Khong thay /kaggle/input/* chua {_MARKER[ds]} cho {ds}')

print(f'DATASETS={DATASETS}  SEED={SEED}  NORM_MODE={NORM_MODE}  MERGE_VAL={MERGE_VAL}')
for ds in DATASETS:
    try:    print(f'  {ds:6s} mount: {resolve_mount(ds)}')
    except Exception as e: print(f'  {ds:6s} !! {e}')

In [ ]:
# Cell 4 — Smoke: TFMamba goc chay duoc tren GPU (fail-fast)
import torch
from model import TFMamba

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_m = TFMamba(num_features=135, num_classes=6, max_len=500).to(dev)
with torch.no_grad():
    _o = _m(torch.randn(2, 500, 135, device=dev), torch.randn(2, 500, 135, device=dev))
assert _o.shape == (2, 6), f'bad output {tuple(_o.shape)}'
print(f'SMOKE OK ({dev})  params={sum(p.numel() for p in _m.parameters())/1e6:.3f}M')
del _m, _o
if dev == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Cell 5 — Chay ca 3 dataset (build DWT in-memory + train + eval) + luu ket qua
import json
from train import run_all

specs = {ds: resolve_mount(ds) for ds in DATASETS}
results = run_all(specs, device='cuda', seed=SEED, norm_mode=NORM_MODE, merge_val=MERGE_VAL)

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('
saved -> /kaggle/working/results.json')

from IPython.display import FileLink, display
display(FileLink('results.json'))